# IMDB report tables generator

This notebook builds the tables you need for the report from:

- `all_model_results.csv`
- `ablation_results.csv`

It also includes a section to compute **dataset length statistics** and **length-bin counts** for the IMDB reviews.

Run top to bottom.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(".")
if not (BASE / "all_model_results.csv").exists():
    BASE = Path("/mnt/data")

all_results = pd.read_csv(BASE / "all_model_results.csv")
ablation_results = pd.read_csv(BASE / "ablation_results.csv")

print("Loaded:")
print(" -", BASE / "all_model_results.csv")
print(" -", BASE / "ablation_results.csv")
print()
print("all_model_results shape:", all_results.shape)
print("ablation_results shape:", ablation_results.shape)


## 1) Dataset length statistics

This cell tries to load the IMDB dataset directly. If that fails in your local environment, replace the loading block with the same dataset-loading code you used in `runner.py`.


In [ ]:
# Try Hugging Face first
texts = None

try:
    from datasets import load_dataset
    ds = load_dataset("imdb")
    texts = list(ds["train"]["text"]) + list(ds["test"]["text"])
    print(f"Loaded IMDB directly from datasets: {len(texts):,} reviews")
except Exception as e:
    print("Direct dataset load failed.")
    print("Replace this block with your runner.py dataset loading code.")
    print("Error:", e)

# Example fallback pattern:
# from runner import <your_loader_function>
# train_texts, train_labels, test_texts, test_labels = <your_loader_function>()
# texts = list(train_texts) + list(test_texts)


In [ ]:
if texts is None:
    raise ValueError("Set `texts` first using your runner.py loading code or the datasets package.")

word_lengths = [len(str(t).split()) for t in texts]

# Token lengths using RoBERTa tokenizer since your report discusses token-based bins
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
token_lengths = [len(tokenizer.encode(str(t), truncation=False)) for t in texts]

def summarize(lengths, name):
    return {
        "Length type": name,
        "Min": int(np.min(lengths)),
        "Median": int(np.median(lengths)),
        "Mean": round(float(np.mean(lengths)), 2),
        "Max": int(np.max(lengths)),
    }

length_stats = pd.DataFrame([
    summarize(word_lengths, "Words"),
    summarize(token_lengths, "Tokens"),
])

length_stats


In [ ]:
def length_bin(token_len):
    if token_len < 128:
        return "short"
    elif token_len <= 512:
        return "medium"
    else:
        return "long"

bin_counts = (
    pd.Series(token_lengths, name="token_length")
      .map(length_bin)
      .value_counts()
      .reindex(["short", "medium", "long"])
      .rename_axis("Length bin")
      .reset_index(name="Review count")
)

bin_counts["Percent"] = (100 * bin_counts["Review count"] / bin_counts["Review count"].sum()).round(2)
bin_counts


## 2) Main model comparison table

This matches the three models you used in your primary comparison figure:
- Logistic Regression
- `BiLSTM_T128_head+tail`
- `RoBERTa_T512_head-only`


In [ ]:
primary_models = [
    "Logistic Regression",
    "BiLSTM_T128_head+tail",
    "RoBERTa_T512_head-only",
]

overall_test_table = (
    all_results[
        (all_results["split"] == "test") &
        (all_results["metric_scope"] == "overall") &
        (all_results["model"].isin(primary_models))
    ][["model", "accuracy", "f1", "auc", "training_time_seconds", "inference_time_seconds"]]
    .copy()
    .sort_values("model")
)

overall_test_table = overall_test_table.rename(columns={
    "model": "Model",
    "accuracy": "Accuracy",
    "f1": "F1",
    "auc": "AUC",
    "training_time_seconds": "Train time (s)",
    "inference_time_seconds": "Inference time (s)",
})

for col in ["Accuracy", "F1", "AUC", "Train time (s)", "Inference time (s)"]:
    overall_test_table[col] = overall_test_table[col].round(4)

overall_test_table


## 3) F1 by review length group

This is the table version of your main length-bin figure.


In [ ]:
f1_by_length_table = (
    all_results[
        (all_results["split"] == "test") &
        (all_results["metric_scope"] == "by_length") &
        (all_results["model"].isin(primary_models))
    ][["model", "length_bin", "f1"]]
    .copy()
    .pivot(index="model", columns="length_bin", values="f1")
    .reindex(primary_models)
    [["short", "medium", "long"]]
    .reset_index()
)

f1_by_length_table.columns = ["Model", "Short F1", "Medium F1", "Long F1"]
for col in ["Short F1", "Medium F1", "Long F1"]:
    f1_by_length_table[col] = f1_by_length_table[col].round(4)

f1_by_length_table


## 4) Ablation tables

These summarize max input length and truncation strategy for BiLSTM and RoBERTa.


In [ ]:
bilstm_ablation = (
    ablation_results[ablation_results["model_family"] == "BiLSTM"]
    [["max_length", "truncation_strategy", "accuracy", "f1", "auc", "training_time_seconds", "inference_time_seconds"]]
    .copy()
    .sort_values(["max_length", "truncation_strategy"])
    .rename(columns={
        "max_length": "Max length",
        "truncation_strategy": "Truncation",
        "accuracy": "Accuracy",
        "f1": "F1",
        "auc": "AUC",
        "training_time_seconds": "Train time (s)",
        "inference_time_seconds": "Inference time (s)",
    })
)

for col in ["Accuracy", "F1", "AUC", "Train time (s)", "Inference time (s)"]:
    bilstm_ablation[col] = bilstm_ablation[col].round(4)

bilstm_ablation


In [ ]:
roberta_ablation = (
    ablation_results[ablation_results["model_family"] == "RoBERTa"]
    [["max_length", "truncation_strategy", "accuracy", "f1", "auc", "training_time_seconds", "inference_time_seconds"]]
    .copy()
    .sort_values(["max_length", "truncation_strategy"])
    .rename(columns={
        "max_length": "Max length",
        "truncation_strategy": "Truncation",
        "accuracy": "Accuracy",
        "f1": "F1",
        "auc": "AUC",
        "training_time_seconds": "Train time (s)",
        "inference_time_seconds": "Inference time (s)",
    })
)

for col in ["Accuracy", "F1", "AUC", "Train time (s)", "Inference time (s)"]:
    roberta_ablation[col] = roberta_ablation[col].round(4)

roberta_ablation


## 5) Compact efficiency table


In [ ]:
efficiency_table = overall_test_table[["Model", "Train time (s)", "Inference time (s)"]].copy()
efficiency_table


## 6) Optional: export all tables to CSV and LaTeX

This writes clean files you can use directly in Overleaf.


In [ ]:
out_dir = BASE / "report_tables"
out_dir.mkdir(exist_ok=True)

tables = {
    "length_stats": length_stats,
    "bin_counts": bin_counts,
    "overall_test_table": overall_test_table,
    "f1_by_length_table": f1_by_length_table,
    "bilstm_ablation_table": bilstm_ablation,
    "roberta_ablation_table": roberta_ablation,
    "efficiency_table": efficiency_table,
}

for name, df in tables.items():
    csv_path = out_dir / f"{name}.csv"
    tex_path = out_dir / f"{name}.tex"
    df.to_csv(csv_path, index=False)
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(df.to_latex(index=False, escape=False))
    print(f"Saved {csv_path}")
    print(f"Saved {tex_path}")


## 7) Suggested tables for the report

You probably only need these four in the paper:

1. **Dataset length statistics**
2. **Length-bin counts**
3. **Overall test performance of the three primary models**
4. **F1 by review length group**

The ablation tables can go in the main text or appendix.
